# how to convert HDF5 DSCOVR EPIC L2 VESDR Granules to CF-Compliant NetCDF

## Summary
This notebook encapsulates the complete workflow for converting DSCOVR EPIC L2 VESDR HDF5 granules to cf-compliant NetCDF4 Level2 and Level3 data files.  The science data in the resulting NetCDF files are much easier access in standard GIS and Earth Science applications such as QGIS, ArcGIS, and Panoply.

### Features Developed:
1. **CF Naming Compliance:** Sanitizes HDF5 variables/attributes into CF-strict legal formats.
2. **Dynamic Projection:** Parses the Sinusoidal central meridian and applies it to the corresponding data in each tile group.
3. **Hierarchical CF Generation:** Creates a native CF-1.8 NetCDF preserving the `tileXX` group structure of the original hdf5 granule.
4. **Global Flattening & Mosaicking:** Combines all tiles into a unified Plate Carrée Lat-Lon grid.
5. **Horizontal Gap Filling:** Accounts for high-latitude pixel widening natively (in Plate Carrée grids) via NumPy smearing.
6. **Overlap Resolution:** Resolves overlapping tile edges by selecting the pixel with the lowest Sensor Zenith Angle (VZA).




### Prerequisites

A free(!) account at https://www.earthdata.nasa.gov/ is needed to login and download the appropriate files.

This notebook was tested last using Python Python 3.14.0

Make sure your environment has the following installed:
```bash
pip install h5py netCDF4 numpy pyproj earthaccess matplotlib
```

### Notebook Author / Affiliation

By Walt Baskin / Atmospheric Science Data Center

### Benefits of converting DSCOVR EPIC Level2 VESDR granules to a CF-Comlpliant Level3 NetCDF 4 file

1. **Strips Away the Complex Sinusoidal Tile Grid**

    The native VESDR dataset distributes global data across fragmented regional sinusoidal blocks. For researchers accustomed to standard global grids, handling these tiles is a significant challenge.

    Merging these tiles into a single, uniform Plate Carrée (Equirectangular / Latitude-Longitude) array allows scientists to immediately slice, index, and subset geographic areas using standard pixel coordinates without writing custom spatial transforming scripts.

2. **Enables Instant GIS and Cloud Data Integration**

    Native HDF5 structures with custom sinusoidal definitions often fail to load correctly in off-the-shelf software.

    Because the NetCDF files are strictly CF-Compliant (Climate and Forecast Metadata Conventions), they will immediately natively open with correct spatial referencing in standard tools like OpenJUMP, QGIS, and ArcGIS.
    Furthermore, CF-compliant NetCDF is the gold standard for cloud-native data stacks, allowing the output to plug directly into tools like xarray, rioxarray, and Zarr for parallel processing on high-performance computing clusters.

3. **Provides Immediate Input for Weather and Climate Models**

    Global Climate Models (GCMs) and Numerical Weather Prediction (NWP) systems almost universally ingest atmospheric and land-surface parameters on a standardized, continuous global lat/lon grid.

    By eliminating the tile-stitching step and standardizing the metadata, your program allows meteorologists and climate modelers to directly feed EPIC’s high-frequency diurnal LAI, SLAI, and FPAR variables into land-surface data assimilation pipelines.

4. **Unlocks Time-Series Analysis of Diurnal Dynamics**

    Analyzing a full day of EPIC data (8 to 12 observations) natively requires managing up to a hundred individual regional tile files.

    Because the notebook produces a single global array per variable, stacked sequentially, researchers can now easily extract a clean temporal profile for any single pixel on Earth. This makes it trivial to plot diurnal vegetation stress curves, track rapid morning dry-down, or monitor cloud-free intervals over heavily forested regions.

5. **Guarantees Data Integrity and Standardization**

    By adhering to CF compliance, this program ensures that essential coordinate variables (like latitude, longitude, and time), data scales, offsets, and missing-value flags are explicitly documented within the file metadata. This eliminates guesswork, prevents calculation errors caused by missing scale factors, and guarantees your processed data remains self-describing and reproducible for other teams.

## 1. Setup

In [ ]:
%pip install h5py netCDF4 numpy pyproj earthaccess matplotlib

In [ ]:
import earthaccess
import h5py
import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import netCDF4 as nc
import os
import re
from pyproj import Transformer, CRS
import warnings

## 2. Search for data using earthaccess

We use `earthaccess` to streamline the login to NASA Earthdata. 

<details open>
  <summary><i>Additional resources about earthaccess</i></summary>

- [*earthaccess* documentation](https://earthaccess.readthedocs.io/en/latest/)

</details>

In [ ]:


earthaccess.login()

In [ ]:
short_name = "DSCOVR_EPIC_L2_VESDR"
version = "02"

results = earthaccess.search_data(
    short_name=short_name,
    version=version,
    provider="LARC_CLOUD",
    temporal=("2024-04-08T18:00:00","2024-04-08T19:00:00")
)

print(f"{len(results)} file(s) found.")
print(results)

## 3. Download data

In [ ]:
#uncomment the following line to download all files found
#downloaded_files = earthaccess.download(results, local_path=".")

#just download the first file in results
downloaded_files = earthaccess.download(results[0], local_path=".")

#The rest of this notebook will only process the first file in the download
input_file=downloaded_files[0]

# Explanation of DSCOVR_EPIC_L2_VESDR_02 granule structure


DSCOVR_EPIC_L2_VESDR HDF5 granules have six tile groups (6 of the 8 tile groups illustrated in figure 2 of DSCOVR_VESDR_SDRG_02.pdf link listed below) containing 1000X1000 data arrays of rotated sinusoidal projections with central meridians at 20, 110, -160 and -70 degree. Four zones are extracted from these maps. Each zone is divided into Northern (block number 0) and Southern (block number 1) tiles. (see https://asdc.larc.nasa.gov/documents/dscovr/DSCOVR_VESDR_SDRG_02.pdf for details)



# 4. Example plots of the tile groups in granule:

### DSCOVR_EPIC_L2_VESDR_02_20240408183250_03.h5



In [ ]:

# --- 1. Construct the Custom Colormap ---
# Extract the 256 RGB colors from the standard 'jet' colormap
jet_cmap = plt.get_cmap('jet', 256)
custom_colors = jet_cmap(np.linspace(0, 1, 256))

# Inject sky blue (R=0.54, G=0.80, B=0.92, Alpha=1.0) into exactly index 148
custom_colors[148] = [0.54, 0.80, 0.92, 1.0]

# Package it back up into a usable Matplotlib Colormap
jet_with_sky_blue_clear_sky = ListedColormap(custom_colors)
# Make sure masked (invalid/fill) pixels are completely transparent
jet_with_sky_blue_clear_sky.set_bad(color='white', alpha=0.0) 


# --- 2. Process and Plot the Data ---
#h5_path = "./DSCOVR_EPIC_L2_VESDR_02_20240408183250_03.h5"
h5_path = input_file

with h5py.File(h5_path, 'r') as f:
    tiles = [k for k in f.keys() if k.startswith('tile') and '19_MAIAC_CloudLWmask' in f[k]]

    tiles.sort()
    
    if not tiles:
        print("No tiles containing '19_MAIAC_CloudLWmask' were found.")
    else:
        print(f"Found {len(tiles)} tiles. Plotting ",h5_path," 19_MAIAC_CloudLWmask variables from all tile groups in the hdf5 file...")
        print("Each tile is identified by its zonal Z (from 0 to 3) and block B (0 or 1) coordinates.")
        print()
        print("for example:")
        print("\ttile20 contains arrays with sinusoidal projections centered at -160 degrees Longitude (zone 2) in the Northern Hemisphere (block 0) ")
        print("\ttile31 contains arrays with sinusoidal projections centered at -70 degrees Longitude (zone 3) in the southern hemisphere (block 1)")

        
        cols = min(3, len(tiles))
        rows = math.ceil(len(tiles) / cols)
        
        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
        if len(tiles) == 1: axes = [axes]
        else: axes = axes.flatten()
        
        for i, tile in enumerate(tiles):
            ax = axes[i]
            data = f[tile]['19_MAIAC_CloudLWmask'][:]
            
            # Mask fill value (240 for 19_MAIAC_CloudLWmask)
            masked_data = np.ma.masked_where(data == 240, data)
            
            # Note the addition of vmin=0 and vmax=255. 
            # This locks the data scale so value 148 always maps perfectly to index 148.
            im = ax.imshow(masked_data, cmap=jet_with_sky_blue_clear_sky, 
                           vmin=0, vmax=255, interpolation='nearest')
            
            ax.set_title(f"{tile} - CloudLWmask\n(Clear Sky = Sky Blue)")
            ax.axis('off')
            
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            
        for j in range(i + 1, len(axes)):
            axes[j].axis('off')
            
        plt.tight_layout()
        plt.show()

## 5. Function to change variable name if necessary to be CF-Compliant

In [ ]:

def sanitize_name(name):
    """
    Cleans a string to be compliant with NetCDF/CF naming conventions.
    - Removes leading two digits and an underscore (e.g., '01_LAI' -> 'LAI')
    - Replaces any remaining illegal characters with an underscore
    - Prepends an underscore if the name still starts with a number
    """
    # 1. Remove leading '\d\d_' from the start of the string
    safe_name = re.sub(r'^\d\d_', '', name)
    
    # 2. Replace illegal characters (spaces, commas, dots, etc.) with underscores
    safe_name = re.sub(r'[^a-zA-Z0-9_]', '_', safe_name)
    
    # 3. CF conventions require names to start with a letter or underscore. 
    # (Checking `if safe_name` prevents an IndexError just in case a name was stripped entirely)
    if safe_name and safe_name[0].isdigit():
        safe_name = '_' + safe_name
        
    return safe_name


with h5py.File(input_file, 'r') as h5_in:
    print("conversion of group and parameter names to cf-compliant names if needed from",input_file)
    for group_name in h5_in.keys():
        if group_name.startswith("tile") and isinstance(h5_in[group_name], h5py.Group):
            h5_grp = h5_in[group_name]
            print("\t",group_name,"-->",sanitize_name(group_name))
            for var_name in h5_grp.keys():
                sanitized_name=sanitize_name(var_name)
                print("\t\t",var_name,"-->",sanitize_name(var_name))

## 6. Hierarchical CF-Compliant NetCDF Generation


In [ ]:
#generate the northing and easting arrays based on the central meridian value in the tileXX group attribute
def get_northing_easting_from_matrix(tile_name, proj_attr):
    if isinstance(proj_attr, bytes): proj_str = proj_attr.decode('utf-8')
    elif isinstance(proj_attr, np.ndarray): proj_str = proj_attr[0].decode('utf-8') if isinstance(proj_attr[0], bytes) else str(proj_attr[0])
    else: proj_str = str(proj_attr)
        
    pixel_match = re.search(r"([\d\.]+)\s*m", proj_str)
    meridian_match = re.search(r"meridian\s+(?:is\s+)?([-\d\.]+)", proj_str)
    
    if not pixel_match or not meridian_match: raise ValueError(f"Could not parse projection string: {proj_str}")
        
    pixel_size = float(pixel_match.group(1))
    lon_0 = float(meridian_match.group(1))
    
    indices = np.arange(1000, dtype=np.float64)
    grid_extent_meters = 1000 * pixel_size
    half_grid_size = grid_extent_meters / 2.0
    
    easting_1d = -half_grid_size + (indices * pixel_size) + (pixel_size / 2.0)
    
    is_southern_hemisphere = tile_name.endswith('1')
    if is_southern_hemisphere: northing_1d = 0.0 - (indices * pixel_size) - (pixel_size / 2.0)
    else: northing_1d = grid_extent_meters - (indices * pixel_size) - (pixel_size / 2.0)
        
    return northing_1d, easting_1d, lon_0, pixel_size

print("defined: (function) def get_northing_easting_from_matrix(tile_name, proj_attr)")
print()
print("This function generates the northing and easting arrays needed by standard GIS applications to plot the projected data in each tile group.")
with h5py.File(input_file, 'r') as h5_in:
    for group_name in h5_in.keys():
        if group_name.startswith("tile") and isinstance(h5_in[group_name], h5py.Group):
            h5_grp = h5_in[group_name]
            print("\t",group_name)
            northing_1d, easting_1d, lon_0, _ = get_northing_easting_from_matrix(group_name, h5_grp.attrs.get("Map projection", ""))
            print("\t\tnorthing: [",*northing_1d[:3],"...",*northing_1d[-3:],"]")
            print("\t\teasting: [",*easting_1d[:3],"...",*easting_1d[-3:],"]")
            print("\t\tlongitude_of_central_meridian:",lon_0)

In [ ]:
def convert_epic_h5_to_cf_nc(input_file, output_file):
    """
    Converts a DSCOVR EPIC L2 VESDR HDF5 granule into a strict CF-1.8 compliant 
    NetCDF4 file while preserving the original hierarchical 'tileXX' group structure.

    Algorithm Highlights:
    ---------------------
    1. Group Preservation: Utilizes NetCDF4's hierarchical group capabilities to 
       mirror the original HDF5 structure natively.
    2. Coordinate Variables: Generates physical 1D arrays ('northing' and 'easting') 
       for each tile, tying the 2D science matrices to absolute spatial dimensions.
    3. Grid Mapping: Defines a standard CF 'sinusoidal' projection variable based on the 
       WGS84 ellipsoid, ensuring geospatial tools (like QGIS/Panoply) project the data 
       accurately without ~15km latitudinal distortion shifts.
    4. Variable-Level Metadata: Translates disconnected global HDF5 attributes 
       (like scale_factor and _FillValue) directly onto the local variables they govern.
    5. CF Type Compliance: Automatically upcasts older unsigned integer types (uint8, uint16) 
       to modern signed integers (int16, int32) to satisfy strict CF checker requirements.

    Parameters:
    -----------
    input_file : str
        Path to the input DSCOVR EPIC HDF5 granule (e.g., 'DSCOVR_EPIC_L2_VESDR_02_...h5').
    output_file : str
        Path where the resulting hierarchical NetCDF4 file will be saved.
    """
    print(f"Opening {input_file} for hierarchical CF conversion...")
    
    with h5py.File(input_file, 'r') as h5_in:
        with nc.Dataset(output_file, 'w', format='NETCDF4') as nc_out:
            scale_factors, fill_values = {}, {}

            # ---------------------------------------------------------
            # 1. Transfer Global Attributes & Cache Scaling Data
            # ---------------------------------------------------------
            for attr_name, attr_value in h5_in.attrs.items():
                safe_attr_name = sanitize_name(attr_name)
                
                # Safely decode bytes/arrays to standard Python strings
                if isinstance(attr_value, bytes): 
                    val = attr_value.decode('utf-8')
                elif isinstance(attr_value, np.ndarray): 
                    val = attr_value[0].decode('utf-8') if attr_value.dtype.kind in ['S', 'O'] and isinstance(attr_value[0], bytes) else attr_value[0] if attr_value.size == 1 else attr_value
                else: 
                    val = attr_value
                    
                nc_out.setncattr(safe_attr_name, val)
                
                # Cache scaling variables to apply to individual datasets later
                if "Scale_factor" in attr_name: scale_factors[attr_name] = val
                if "Fill_value" in attr_name: fill_values[attr_name] = val

            # Declare the explicit standard we are adhering to
            nc_out.setncattr("Conventions", "CF-1.8")

            # Helper function: Translates global HDF5 scaling rules to specific dataset names
            def get_cf_scaling(var_name):
                scale, fill = 1.0, fill_values.get("Fill_value_VESDR", -9999) if any(x in var_name for x in ["LAI", "FPAR", "NDVI", "DASF", "QA", "Dlai"]) else fill_values.get("Fill_value_map", -9999)
                if any(x in var_name for x in ["LAI", "FPAR", "NDVI", "DASF", "Dlai"]): scale = scale_factors.get("Scale_factor_VESDR", 1.0)
                elif any(x in var_name for x in ["SZA", "VZA", "SAA", "VAA"]): scale = scale_factors.get("Scale_factor_angle", 1.0)
                elif "AOD" in var_name: scale = scale_factors.get("Scale_factor_AOD", 1.0)
                elif "W" in var_name and not "LW" in var_name: scale = scale_factors.get("Scale_factor_W", 1.0)
                elif "ERTI" in var_name: scale = scale_factors.get("Scale_factor_ERTI", 1.0)
                return scale, fill

            # ---------------------------------------------------------
            # 2. Iterate Over Tiles and Build Hierarchical Groups
            # ---------------------------------------------------------
            for group_name in h5_in.keys():
                if group_name.startswith("tile") and isinstance(h5_in[group_name], h5py.Group):
                    h5_grp = h5_in[group_name]
                    
                    # Create the localized NetCDF Group
                    nc_grp = nc_out.createGroup(sanitize_name(group_name))
                    
                    # Define spatial dimensions specific to this tile (1000x1000 pixels)
                    nc_grp.createDimension('northing', 1000)
                    nc_grp.createDimension('easting', 1000)
                    
                    # Extract the projection string and calculate the 1D physical arrays
                    proj_attr_raw = h5_grp.attrs.get("Map projection", "")
                    northing_1d, easting_1d, lon_0, _ = get_northing_easting_from_matrix(group_name, proj_attr_raw)
                    
                    # --- Create CF Standard Coordinate Variables ---
                    north_var = nc_grp.createVariable('northing', 'f8', ('northing',))
                    north_var.standard_name, north_var.units = 'projection_y_coordinate', 'm'
                    north_var[:] = northing_1d
                    
                    east_var = nc_grp.createVariable('easting', 'f8', ('easting',))
                    east_var.standard_name, east_var.units = 'projection_x_coordinate', 'm'
                    east_var[:] = easting_1d

                    # --- Create CF Standard Grid Mapping (CRS) Dummy Variable ---
                    crs_var = nc_grp.createVariable('sinusoidal', 'i4')
                    crs_var.grid_mapping_name = 'sinusoidal'
                    crs_var.longitude_of_central_meridian = lon_0
                    crs_var.semi_major_axis = 6378137.0         # Defines WGS84 Ellipsoid
                    crs_var.inverse_flattening = 298.257223563  # Defines WGS84 Ellipsoid

                    # Transfer group-level spatial attributes natively
                    for g_attr, g_val in h5_grp.attrs.items():
                        if isinstance(g_val, bytes): g_val = g_val.decode('utf-8')
                        elif isinstance(g_val, np.ndarray) and g_val.dtype.kind in ['S', 'O']: g_val = g_val[0].decode('utf-8') if isinstance(g_val[0], bytes) else str(g_val[0])
                        nc_grp.setncattr(sanitize_name(g_attr), g_val)

                    # ---------------------------------------------------------
                    # 3. Process and Copy the Science Variables
                    # ---------------------------------------------------------
                    for var_name in h5_grp.keys():
                        h5_var = h5_grp[var_name]
                        if isinstance(h5_var, h5py.Dataset):
                            scale, fill = get_cf_scaling(var_name)
                            
                            # Standardize byte order to machine-native to avoid Endian warnings
                            native_dtype = h5_var.dtype.newbyteorder('=')
                            
                            # --- CF Compliance: Upcast Unsigned Integers to Signed ---
                            if native_dtype == np.uint8:
                                cf_dtype = np.dtype('int16')
                            elif native_dtype == np.uint16:
                                cf_dtype = np.dtype('int32')
                            elif native_dtype == np.uint32:
                                cf_dtype = np.dtype('int64')
                            else:
                                cf_dtype = native_dtype
                                
                            try:
                                casted_fill = np.array(fill).astype(cf_dtype)
                            except ValueError:
                                casted_fill = fill
                            # ---------------------------------------------------------
                            
                            # Initialize the NetCDF variable using the compliant CF dtype
                            try: 
                                nc_var = nc_grp.createVariable(sanitize_name(var_name), cf_dtype, ('northing', 'easting'), fill_value=casted_fill, zlib=True)
                            except Exception: 
                                nc_var = nc_grp.createVariable(sanitize_name(var_name), cf_dtype, ('northing', 'easting'), zlib=True)

                            # Suppress internal C-Library deprecation warnings for np.array shape assignments
                            with warnings.catch_warnings():
                                warnings.simplefilter("ignore", category=DeprecationWarning)
                                # Extract data, apply typecast, and write to NetCDF
                                nc_var[:] = np.array(h5_var[:], dtype=cf_dtype)
                            
                            # Bind scaling attributes to the variable level
                            if scale != 1.0: 
                                nc_var.scale_factor = float(scale)
                            
                            # Explicitly tie the data matrix to the 1D Coordinates & WGS84 CRS
                            nc_var.original_name = var_name
                            nc_var.coordinates = "northing easting"
                            nc_var.grid_mapping = "sinusoidal"

                            # Enhance metadata with established CF standard terminology
                            if "SZA" in var_name:
                                nc_var.standard_name, nc_var.units = "solar_zenith_angle", "degrees"
                            elif "VZA" in var_name:
                                nc_var.standard_name, nc_var.units = "sensor_zenith_angle", "degrees"
                            elif "LAI" in var_name and "SLAI" not in var_name and "Dlai" not in var_name:
                                nc_var.standard_name, nc_var.units = "leaf_area_index", "1"
                            elif "NDVI" in var_name:
                                nc_var.standard_name, nc_var.units = "normalized_difference_vegetation_index", "1"
                            elif "AOD" in var_name:
                                nc_var.standard_name, nc_var.units = "atmosphere_optical_thickness_due_to_aerosol", "1"

    print(f"Success! CF-compliant Hierarchical NetCDF saved to: {output_file}")


print("defined: (function) def convert_epic_h5_to_cf_nc(input_file, output_file)")
print("\tThis hierarchical CF conversion function converts the original HDF5 file to NetCDF4 while preserving the group structure of the original file...")
print()
print("The following variables are added to each tile group of the output NetCDF file to explicity provide the necessary information for standard GIS applications to interpret the different projections in each tile group.")
print("\tnorthing")
print("\teasting")
print("\tsinusoidal")

## 7. Flat Plate Carrée Reprojection & Gap Filling


In [ ]:
def parse_h5_attrs(h5_obj):
    parsed = {}
    for k, v in h5_obj.attrs.items():
        if isinstance(v, bytes): parsed[k] = v.decode('utf-8')
        elif isinstance(v, np.ndarray):
            if v.dtype.kind in ['S', 'O']: parsed[k] = v[0].decode('utf-8') if isinstance(v[0], bytes) else str(v[0])
            else: parsed[k] = v[0] if v.size == 1 else v
        else: parsed[k] = v
    return parsed

print("defined: (function) def parse_h5_attrs(h5_obj) -> dict")

def merge_and_reproject_to_flat_nc(input_file, output_file, resolution=0.1):
    """
    Reads a multi-tile DSCOVR EPIC L2 VESDR HDF5 granule, reprojects the Sinusoidal 
    data to a global Plate Carrée (Equirectangular) grid, and exports a CF-compliant NetCDF4 file.

    Algorithm Highlights:
    ---------------------
    1. Reprojection: Uses WGS84 Ellipsoidal Sinusoidal projection math to map tile Cartesian 
       coordinates (Northing/Easting) into absolute geographic Latitudes and Longitudes.
    2. Data Flattening: Strips out space/fill-value pixels early to drastically reduce RAM usage, 
       flattening all valid tile data into single 1D arrays.
    3. Gap Filling (Horizontal Smearing): Sinusoidal equal-area pixels undergo extreme horizontal 
       stretching near the poles when mapped to a Lat/Lon grid. This function calculates the 
       physical width of each pixel based on the cosine of its latitude and duplicates the pixel 
       horizontally to prevent visual gaps.
    4. Overlap Resolution: When adjacent satellite swaths or tiles overlap, the function uses 
       the Sensor Zenith Angle (VZA) as a tie-breaker, preserving only the pixel with the lowest 
       VZA (least optical distortion).

    Parameters:
    -----------
    input_file : str
        Path to the input DSCOVR EPIC HDF5 granule (e.g., 'DSCOVR_EPIC_L2_VESDR_02_...h5').
    output_file : str
        Path where the resulting flat NetCDF4 file will be saved.
    resolution : float, optional
        The degree resolution of the target Plate Carrée grid. Default is 0.1 degrees 
        (roughly equivalent to the native 10km EPIC resolution at the equator).
    """
    print(f"Flattening and Reprojecting {input_file} to {output_file}...")
    
    # Target Coordinate Reference System (WGS84 Lat/Lon)
    wgs84_crs = CRS.from_epsg(4326)
    
    # ---------------------------------------------------------
    # 1. Setup Gridding Target Matrix (Plate Carrée)
    # ---------------------------------------------------------
    lat_min, lat_max = -90.0, 90.0
    lon_min, lon_max = -180.0, 180.0
    
    # Calculate grid dimensions based on the requested resolution
    n_lat = int(np.round((lat_max - lat_min) / resolution)) 
    n_lon = int(np.round((lon_max - lon_min) / resolution)) 
    
    # Generate 1D coordinate arrays for the cell centers
    lats_1d = lat_min + (np.arange(n_lat) * resolution) + (resolution / 2.0)
    lons_1d = lon_min + (np.arange(n_lon) * resolution) + (resolution / 2.0)
    
    # Master lists to accumulate flattened arrays from all tiles
    master_lons, master_lats, master_vza = [], [], []
    master_vars = {}
    
    with h5py.File(input_file, 'r') as h5_in:
        # Extract root attributes for global NetCDF metadata and dynamic scaling
        root_attrs = parse_h5_attrs(h5_in)
        fill_values = {k: v for k, v in root_attrs.items() if "Fill_value" in k}
        scale_factors = {k: v for k, v in root_attrs.items() if "Scale_factor" in k}
        
        # Discover available spatial tiles and science variables
        tile_keys = [k for k in h5_in.keys() if 'tile' in k.lower() and isinstance(h5_in[k], h5py.Group)]
        var_names = [k for k in h5_in[tile_keys[0]].keys() if isinstance(h5_in[tile_keys[0]][k], h5py.Dataset)]
        for v in var_names: 
            master_vars[v] = []
            
        vza_fill = fill_values.get("Fill_value_map", -9999)
        global_pixel_size = 10000.0 # Fallback 10km grid
        
        # ---------------------------------------------------------
        # 2. Extract, Reproject, and Flatten Tile Data
        # ---------------------------------------------------------
        for tile in tile_keys:
            group = h5_in[tile]
            if '08_VZA' not in group: continue
            
            vza = group['08_VZA'][:]
            proj_attr = group.attrs.get("Map projection")
            northing_1d, easting_1d, lon_0, tile_px_size = get_northing_easting_from_matrix(tile, proj_attr)
            global_pixel_size = tile_px_size 
            
            # Generate the Cartesian 2D grid for the current tile
            X, Y = np.meshgrid(easting_1d, northing_1d) 
            
            # WGS84 Ellipsoid parameter enforcement (+datum=WGS84) is critical here 
            # to prevent ~15km latitudinal shifts in coastline rendering.
            sinu_crs = f"+proj=sinu +lon_0={lon_0} +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
            transformer = Transformer.from_crs(sinu_crs, wgs84_crs, always_xy=True)
            lons, lats = transformer.transform(X, Y)
            
            # Strip invalid pixels before concatenation to drastically reduce memory usage
            valid_mask_flat = ((vza != vza_fill) & (vza >= 0.0) & (vza <= 90.0)).flatten()
            
            master_lons.append(lons.flatten()[valid_mask_flat])
            master_lats.append(lats.flatten()[valid_mask_flat])
            master_vza.append(vza.flatten()[valid_mask_flat])
            
            for v in var_names: 
                master_vars[v].append(group[v][:].flatten()[valid_mask_flat])

    # Combine accumulated tile fragments into massive, unified 1D vectors
    all_lons = np.concatenate(master_lons)
    all_lats = np.concatenate(master_lats)
    all_vzas = np.concatenate(master_vza)
    
    # ---------------------------------------------------------
    # 3. Horizontal Pixel Smearing (High Latitude Gap Filler)
    # ---------------------------------------------------------
    # Because Sinusoidal is an Equal-Area projection, pixels physically widen in degrees 
    # the further they are from the equator. We calculate how many Plate Carrée grid 
    # cells each pixel should occupy based on its Latitude.
    meters_per_deg_eq = np.pi * 6378137.0 / 180.0 # WGS84 Semi-Major Axis 
    cos_lats = np.clip(np.cos(np.radians(all_lats)), 1e-5, 1.0) # Prevent divide-by-zero at poles
    delta_lon_deg = (global_pixel_size / meters_per_deg_eq) / cos_lats
    
    # Determine start and end column indices for each smeared pixel
    start_lon_idx = np.floor((all_lons - (delta_lon_deg / 2.0) - lon_min) / resolution).astype(int)
    lon_counts = np.maximum(1, np.floor((all_lons + (delta_lon_deg / 2.0) - lon_min) / resolution).astype(int) - start_lon_idx + 1)
    
    # Use NumPy's highly optimized `repeat` to clone the data points horizontally
    expanded_pixel_ids = np.repeat(np.arange(len(all_lons)), lon_counts)
    
    # Generate the target geographic indices for the newly smeared pixels
    expanded_lon_idx = (start_lon_idx[expanded_pixel_ids] + np.arange(lon_counts.sum()) - np.repeat(lon_counts.cumsum() - lon_counts, lon_counts)) % n_lon
    expanded_lat_idx = np.clip(np.floor((all_lats - lat_min) / resolution).astype(int), 0, n_lat - 1)[expanded_pixel_ids]
    
    # Create a 1D mapping index corresponding to the target (lat, lon) matrix
    expanded_flat_grid_idx = expanded_lat_idx * n_lon + expanded_lon_idx
    
    # ---------------------------------------------------------
    # 4. Resolve Overlaps (Tie-Breaker: Minimum VZA)
    # ---------------------------------------------------------
    # Because satellite swaths overlap, multiple pixels may attempt to write to the same 
    # Plate Carrée cell. By sorting by Sensor Zenith Angle (VZA) first, the least-distorted 
    # (most nadir) pixel is guaranteed to appear first in the array.
    sort_order = np.argsort(all_vzas[expanded_pixel_ids])
    
    # `np.unique` naturally returns the index of the first occurrence it sees.
    _, unique_first_indices = np.unique(expanded_flat_grid_idx[sort_order], return_index=True)
    
    # Trace the winning indices back to their geographic and source data locations
    best_expanded_idx = sort_order[unique_first_indices]
    best_target_flat_idx = expanded_flat_grid_idx[best_expanded_idx]
    
    # Decode the 1D mapping index back into explicit matrix Row (Lat) and Column (Lon) 
    best_lat_idx, best_lon_idx = best_target_flat_idx // n_lon, best_target_flat_idx % n_lon
    best_source_pixel_ids = expanded_pixel_ids[best_expanded_idx]

    # ---------------------------------------------------------
    # 5. Write the Output File (CF Conventions)
    # ---------------------------------------------------------
    with nc.Dataset(output_file, 'w', format='NETCDF4') as nc_out:
        
        # Define dimensions standard (Row = latitude, Col = longitude)
        nc_out.createDimension('latitude', n_lat)
        nc_out.createDimension('longitude', n_lon)
        
        # Latitude Coordinate Variable
        lat_var = nc_out.createVariable('latitude', 'f8', ('latitude',))
        lat_var.standard_name, lat_var.units = 'latitude', 'degrees_north'
        lat_var[:] = lats_1d
        
        # Longitude Coordinate Variable
        lon_var = nc_out.createVariable('longitude', 'f8', ('longitude',))
        lon_var.standard_name, lon_var.units = 'longitude', 'degrees_east'
        lon_var[:] = lons_1d
        
        # Transfer sanitized global attributes
        for k, v in root_attrs.items(): 
            nc_out.setncattr(sanitize_name(k), v)
        
        nc_out.setncattr("Conventions", "CF-1.8")

        # Process and write each individual science variable
        for var_name in var_names:
            # Determine appropriate default fill value
            v_fill = fill_values.get("Fill_value_VESDR", -9999) if any(x in var_name for x in ["LAI", "FPAR", "NDVI", "DASF", "QA", "Dlai"]) else fill_values.get("Fill_value_map", -9999)
            
            # Fetch the winning source pixels from the flattened array
            best_var_data = np.concatenate(master_vars[var_name])[best_source_pixel_ids]
            
            # Initialize a blank target matrix filled with the designated fill_value
            try: 
                grid_2d = np.full((n_lat, n_lon), fill_value=np.array(v_fill).astype(best_var_data.dtype), dtype=best_var_data.dtype)
            except ValueError: 
                grid_2d = np.full((n_lat, n_lon), fill_value=v_fill, dtype=best_var_data.dtype)

            # Insert the winning valid data into the geographic matrix
            grid_2d[best_lat_idx, best_lon_idx] = best_var_data
            
            # Create CF-compliant variable mapped explicitly to ('latitude', 'longitude')
            try: 
                nc_var = nc_out.createVariable(sanitize_name(var_name), best_var_data.dtype, ('latitude', 'longitude'), fill_value=np.array(v_fill).astype(best_var_data.dtype), zlib=True)
            except Exception: 
                nc_var = nc_out.createVariable(sanitize_name(var_name), best_var_data.dtype, ('latitude', 'longitude'), zlib=True)

            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=DeprecationWarning)
                nc_var[:] = grid_2d
            
            # Apply dynamic CF scaling variables if scale factor != 1.0
            scale = scale_factors.get("Scale_factor_VESDR", 1.0) if any(x in var_name for x in ["LAI", "FPAR", "NDVI", "DASF", "Dlai"]) else scale_factors.get("Scale_factor_angle", 1.0) if any(x in var_name for x in ["SZA", "VZA", "SAA", "VAA"]) else scale_factors.get("Scale_factor_AOD", 1.0) if "AOD" in var_name else scale_factors.get("Scale_factor_W", 1.0) if "W" in var_name and not "LW" in var_name else scale_factors.get("Scale_factor_ERTI", 1.0) if "ERTI" in var_name else 1.0
            if scale != 1.0: 
                nc_var.scale_factor = float(scale)
                
            nc_var.original_name = var_name
            nc_var.coordinates = "latitude longitude"

    print(f"Success! Gap-filled Plate Carrée NetCDF saved to: {output_file}")

print("defined: (function) def merge_and_reproject_to_flat_nc(input_file, output_file, resolution=0.1))")


## 8. Execution
## Generate: CF-Compliant NetCDF files from the original HDF5 granule
### **DSCOVR_EPIC_L2_VESDR_02_20240408183250_03.nc** 
    Retains the same schema and arrays as orginal files with the addition of northing, easting, and sinusoidal parameters for each tile group

### **DSCOVR_EPIC_L3_VESDR_02_20240408183250_03.nc** 
    All parameters from each tile group are merged into a single 1800X3600 Plate Carrée array at the root level of the resulting NetCDF file

In [ ]:
hier_out = str.replace(str(input_file),".h5",".nc")
flat_out = str.replace(hier_out,"_L2_","_L3_")

if os.path.exists(input_file):
    convert_epic_h5_to_cf_nc(input_file, hier_out)
    merge_and_reproject_to_flat_nc(input_file, flat_out, resolution=0.1)
else:
    print(f"File not found: {input_file}")


               
## 9. Plot DSCOVR_EPIC_L3_VESDR_02_20240408183250_03.nc MAIAC_CloudLWmask variable
The pixel resolution in the plot below from the 1800x3600 MAIAC_CloudLWmask array is approximately 10X10 km at the equator.

In [ ]:

# 1. Define the File Path
#flat_out = "./DSCOVR_EPIC_L3_VESDR_02_20240408183250_03.nc"
# flat_out is defined in previous cell

if not os.path.exists(flat_out):
    print(f"Error: {flat_out} not found. Please run the flattening script first.")
else:
    # 2. Construct the Custom Colormap
    jet_cmap = plt.get_cmap('jet', 256)
    custom_colors = jet_cmap(np.linspace(0, 1, 256))
    
    # Inject sky blue (R=0.54, G=0.80, B=0.92, Alpha=1.0) into exactly index 148
    custom_colors[148] = [0.54, 0.80, 0.92, 1.0]
    
    # Rebuild and set missing/fill data to transparent
    jet_with_blue_clear_sky = ListedColormap(custom_colors)
    jet_with_blue_clear_sky.set_bad(color='white', alpha=0.0) 

    # 3. Open the Flat NetCDF File
    with nc.Dataset(flat_out, 'r') as ds:
        
        # Using the sanitized CF-compliant variable name
        var_name = 'MAIAC_CloudLWmask' 
        
        if var_name not in ds.variables:
            print(f"Error: {var_name} not found in the NetCDF variables.")
        else:
            print("Compare the spatial coverage of the clear sky extent (light blue) in the Plate Carrée plot below")
            print("with the clear sky extents of the sinusoidal projection plots from the original hdf5 granule:")
            print(input_file)
            # Extract geographic extents
            lons = ds['longitude'][:]
            lats = ds['latitude'][:]
            extent = [lons.min(), lons.max(), lats.min(), lats.max()]

            # Extract the masked science data array
            data = ds[var_name][:]
            
            # 4. Create the Plot
            plt.figure(figsize=(12, 7))
            
            # Lock vmin=0 and vmax=255 so the value 148 aligns perfectly with color index 148
            im = plt.imshow(data, cmap=jet_with_blue_clear_sky, origin='lower', 
                            extent=extent, vmin=0, vmax=255, interpolation='nearest')
            
            # Formatting
            plt.title("DSCOVR EPIC - MAIAC Cloud/Land/Water Mask\n(Value 148 = Sky Blue Clear Sky)", fontsize=14)
            plt.xlabel("Longitude", fontsize=12)
            plt.ylabel("Latitude", fontsize=12)
            plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
            
            # Add Colorbar
            cbar = plt.colorbar(im, fraction=0.036, pad=0.04)
            cbar.set_label("Mask Value", fontsize=12)
            
            plt.tight_layout()
            plt.show()

## 10. Plot all the variables in the flattened, DSCOVR_EPIC_L2_VESDR_02 NetCDF file

In [ ]:
#Plot all the variables in the flattened, DSCOVR_EPIC_L2_VESDR_02 NetCDF file created by the merge_and_reproject_to_flat_nc function

# 1. Define File Paths
#flat_out = "./DSCOVR_EPIC_L3_VESDR_02_20240408183250_03.nc"
# flat_out is defined in a previous cell

if not os.path.exists(flat_out):
    print(f"Error: {flat_out} not found. Please run the generation script first.")
else:


    # 2. Open the Flat NetCDF File
    with nc.Dataset(flat_out, 'r') as ds:
        # Extract dynamic extents from coordinate arrays for proper geographic plotting
        lons = ds['longitude'][:]
        lats = ds['latitude'][:]
        extent = [lons.min(), lons.max(), lats.min(), lats.max()]

        # Filter out coordinate/mapping variables to get only the science variables
        skip_vars = ['latitude', 'longitude', 'sinusoidal']
        var_names = [v for v in ds.variables if v not in skip_vars]
        
        print(f"Found {len(var_names)} variables to plot...")

        # 3. Set up the Subplot Grid
        cols = 3
        rows = math.ceil(len(var_names) / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
        axes = axes.flatten()
        
        # 4. Loop through and plot all variables
        for i, var_name in enumerate(var_names):
            ax = axes[i]
            
            # Extract data. NetCDF4 automatically applies scale_factor and _FillValue masks!
            data = ds[var_name][:]
            

            cmap = 'turbo' # Modern, colorful default for other variables
            vmin, vmax = None, None # Let matplotlib auto-scale the data

            # Plot using origin='lower' since latitudes ascend from -90 to +90
            im = ax.imshow(data, cmap=cmap, origin='lower', extent=extent, 
                           vmin=vmin, vmax=vmax, interpolation='nearest')
            
            # Try to grab the CF standard name for a cleaner title, fallback to variable name
            title = getattr(ds[var_name], 'standard_name', var_name).replace('_', ' ').title()
            ax.set_title(f"{var_name}\n({title})", fontsize=10)
            
            # Add gridlines and axis labels
            ax.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")
            
            # Add colorbar (append units if available)
            units = getattr(ds[var_name], 'units', '')
            cbar_label = f"[{units}]" if units else ""
            fig.colorbar(im, ax=ax, fraction=0.036, pad=0.04, label=cbar_label)
            
        # Hide any unused subplots
        for j in range(i + 1, len(axes)):
            axes[j].axis('off')
            
        plt.tight_layout()
        plt.show()